In [7]:
# For Google Colab. Remove or
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")

    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [9]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.39G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.3.3 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [11]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="phi-4",
)

In [12]:
# SYSTEM_PROMPT = """
# You are an expert in parsing terminal session logs.

# Process the provided <system_output> starting from the very first line. You must be extremely careful to examine every single line sequentially without skipping.

# An event boundary always occurs between a user's input and the subsequent system output. Identify the absolute first occurrence of a shell prompt (e.g., demo@boxtop:~$ or similar) which signifies that the previous command has finished and a new event is starting.

# For each line, if it marks the start of a new event, you must return exactly one timestamp from that prompt line to define the end of the current group.

# Do not return any text, explanation, or multiple lines—only the single target timestamp.

# Return format:
# <timestamp>
# """
SYSTEM_PROMPT = """
You are an expert in parsing terminal session logs.

Extract **only the timestamps** where a shell prompt appears inside <system_output>.
Do not return any text, explanation, or formatting — only timestamps, one per line.

A shell prompt looks like:
demo@boxtop:~$
or any similar prompt indicating that the previous command finished.

Each timestamp you return **defines an event boundary**.
Remember: Only return timestamps. Each timestamp defines an event boundary. No extra text or explanation.
***Important note: The switch between the user input and system output should not be treated as a boundary.***

Return format:
<timestamp>
"""
xml_data = """
{"input": "<system_output timestamp=\"0.008584\">[?2004hdemo@stephost:~$ </system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"0.603025\"/>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"0.603588\">\n(reverse-i-search)`': </system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"0.937684\">s</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"0.938181\">s': [7ms[27mync</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"1.114177\">s</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"1.1146\">s': [7mss[27mh 172.16.0.17</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"1.291778\">h</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"1.292225\">[1@h': [7mssh[27m</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"2.617004\">\n</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"2.617465\">\n[8Pdemo@stephost:~$ ssh\n[?2004l\n</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"5.694771\">ssh: connect to host 172.16.0.17 port 22: No route to host\n\n</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"5.697578\">[?2004hdemo@stephost:~$ </system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"7.138774\">c</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"7.139172\">c</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"7.367648\">m</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"7.368275\">m</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"7.491765\"> </user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"7.492239\"> </system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"7.700208\">l</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"7.700707\">l</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"7.826627\">i</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"7.827056\">i</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"8.079826\">s</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.080258\">s</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"8.28657\">t</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.286967\">t</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"8.474631\">\n</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.475097\">\n[?2004l\n</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.48351\">implicitserver-tearoff-16: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.523754\">\nimplicitserver-tearoff-17: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.575367\">\nwikiserver-tearoff-3: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.604153\"> Running\nqemuwordserver-tearoff-4: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.63601\">\nlampserver-tearoff-14: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.667641\">\nwikiserver-tearoff-5: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.699481\">\ndrupalserver-tearoff-13: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.730786\">\nnullhost-tearoff-11: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.76265\">\nwikiserver-tearoff-2: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.794868\">\nfaiserver-tearoff-8: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.826481\">\nfaiserver-tearoff-9: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.858307\">\nfaiserver-tearoff-1: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.888755\">\nfaiserver-tearoff-19: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.918944\">\nnullhost-tearoff-10: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.952225\"> Running\nnullhost-tearoff-7: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.983719\">\nfaiserver-tearoff-18: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"9.015378\">\nfaiserver-tearoff-12: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"9.047348\">\nnullhost-tearoff-6: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"9.079908\"> Running\nnullhost-tearoff-15: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"9.110634\">\n</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"9.110867\">[?2004hdemo@stephost:~$ </system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"15.55465\"/>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"15.555045\">[?2004l\n\nexit\n</system_output>\n\n[Script Action: Processed XML chunk.]"}
"""

In [13]:
# === Run inference (assumes `model` and `tokenizer` are already loaded) ===
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Extract event boundaries from this XML:\n{xml_data}"}
]

# Tokenize & move to GPU
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate from model
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.2,
)

import re

# raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]
raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]

# extract the assistant block using the marker tokens, if present
m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output, re.S)
if m:
    raw_assistant = m.group(1).strip()
else:
    # fallback to full decode if markers missing
    raw_assistant = raw_output

print("=== RAW ASSISTANT OUTPUT ===")
print(raw_assistant)

# same extraction as above
timestamps = re.findall(r'\d+\.\d+', raw_assistant)
# dedupe-preserve-order
seen = set(); ordered = []
for t in timestamps:
    if t not in seen:
        seen.add(t); ordered.append(t)

print("\n=== CLEANED EXTRACTED TIMESTAMPS (assistant-only) ===")
print("\n".join(ordered) if ordered else "(no timestamps found)")


# === Print clearly separated input and model "true" output ===
print("=== INPUT XML ===")
print(xml_data)
print("\n=== RAW MODEL OUTPUT (decoded) ===")
print(raw_output)
print("\n=== CLEANED EXTRACTED TIMESTAMPS ===")
if timestamps:
    print("\n".join(timestamps))
else:
    print("(no timestamps found)")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== RAW ASSISTANT OUTPUT ===
0.008584  
2.617465  
5.697578  
8.475097  
9.110867

=== CLEANED EXTRACTED TIMESTAMPS (assistant-only) ===
0.008584
2.617465
5.697578
8.475097
9.110867
=== INPUT XML ===

{"input": "<system_output timestamp="0.008584">[?2004hdemo@stephost:~$ </system_output>

[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp="0.603025"/>

[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp="0.603588">
(reverse-i-search)`': </system_output>

[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp="0.937684">s</user_input>

[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp="0.938181">s': sync</system_output>

[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp="1.114177">s</user_input>

[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp="1.1146">s': ssh 172.16.0.17</system_output>

[Script Action: Processed XML chunk.]"}
{"input": "<user_inp

In [18]:
!pip install -q scikit-learn

In [26]:
# Basic F1 and WindowDiff metric evaluation of model 0
# Limitations: truncates input files until they fit the model context window

import os, re, json
import numpy as np
from sklearn.metrics import f1_score

INPUT_DIR  = "/content/drive/MyDrive/Colab Notebooks/AutoDocs/data-processing/model_0/inputs"
LABEL_DIR = "/content/drive/MyDrive/Colab Notebooks/AutoDocs/data-processing/model_0/timestamp-output"
MAX_CHUNKS = 100  # truncate large files to fit in context

# -- helpers --

def parse_rec_xml(path):
    """Return list of (timestamp_float, xml_chunk_str) for each element."""
    chunks = []
    with open(path, encoding='utf-8', errors='ignore') as f:
        text = f.read()
    pat = re.compile(
        r'<(system_output|user_input)\s+timestamp="([\d.]+)"(?:\s*/>|>(.*?)</\1>)',
        re.DOTALL)
    for m in pat.finditer(text):
        tag, ts_str, body = m.group(1), m.group(2), m.group(3) or ''
        xml_str = (f'<{tag} timestamp="{ts_str}">{body}</{tag}>'
                   if body else f'<{tag} timestamp="{ts_str}"/>')
        chunks.append((float(ts_str), xml_str))
    return chunks


def build_prompt(chunks):
    """Build the same JSONL prompt format used in model0_without_finetune.ipynb."""
    lines = []
    for _, xml_str in chunks:
        payload = xml_str + '\n\n[Script Action: Processed XML chunk.]'
        lines.append(json.dumps({"input": payload}))
    return '\n'.join(lines)


def run_inference(chunks):
    max_input_tokens = max_seq_length - 256  # reserve for generation

    # trim from the end until the prompt fits
    while chunks:
        prompt = build_prompt(chunks)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Extract event boundaries from this XML:\n{prompt}"}
        ]
        inp = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")
        if inp.shape[-1] <= max_input_tokens:
            break
        chunks = chunks[:-1]

    if not chunks:
        return []

    out = model.generate(input_ids=inp, max_new_tokens=256, use_cache=True, temperature=0.2)
    raw = tokenizer.batch_decode(out, clean_up_tokenization_spaces=True)[0]
    m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)(?:<\|im_end\|>|$)', raw, re.S)
    assistant = m.group(1).strip() if m else raw
    seen, pred = set(), []
    for t in re.findall(r'\d+\.\d+', assistant):
        if t not in seen:
            seen.add(t)
            pred.append(float(t))
    return pred



def ts_to_binary(chunk_ts, target_ts, tol=0.5):
    """Per-chunk binary vector: 1 where nearest chunk timestamp matches a target."""
    vec = [0] * len(chunk_ts)
    for t in target_ts:
        diffs = [abs(ct - t) for ct in chunk_ts]
        idx = int(np.argmin(diffs))
        if diffs[idx] <= tol:
            vec[idx] = 1
    return vec


def windowdiff(ref, hyp, k):
    """Pevzner & Hearst WindowDiff. Lower = better; 0 = perfect."""
    N = len(ref)
    if N <= k:
        return 0.0
    errors = sum(
        abs(sum(ref[i+1:i+k+1]) - sum(hyp[i+1:i+k+1])) > 0
        for i in range(N - k)
    )
    return errors / (N - k)


# -- evaluation loop --

from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

results = []

for fname in sorted(os.listdir(INPUT_DIR)):

    # End evaluation early if needed
    if len(results) >= 10:
      break

    # strip known recording extensions to get the base name
    base = fname
    for ext in ('.rec.xml', '.asciinema.xml', '.cast.xml'):
        if fname.endswith(ext):
            base = fname[:-len(ext)]
            break

    label_path = os.path.join(LABEL_DIR, f"{base}.time.txt")
    if not os.path.exists(label_path):
        print(f"[skip] no label file for {fname}")
        continue

    # load ground-truth boundary timestamps
    gt_ts = []
    with open(label_path) as lf:
        for line in lf:
            line = line.strip()
            if line:
                try:
                    gt_ts.append(float(line))
                except ValueError:
                    pass

    chunks = parse_rec_xml(os.path.join(INPUT_DIR, fname))
    if not chunks or not gt_ts:
        print(f"[skip] empty data for {fname}")
        continue

    if len(chunks) > MAX_CHUNKS:
        print(f"[warn] {fname}: {len(chunks)} chunks, truncating to {MAX_CHUNKS}")
        chunks = chunks[:MAX_CHUNKS]

    chunk_ts = [ts for ts, _ in chunks]
    pred_ts  = run_inference(chunks)

    ref = ts_to_binary(chunk_ts, gt_ts)
    hyp = ts_to_binary(chunk_ts, pred_ts)

    n_seg = max(1, sum(ref))
    k     = max(1, len(ref) // (2 * n_seg))
    f1    = f1_score(ref, hyp, pos_label=1, zero_division=0)
    wd    = windowdiff(ref, hyp, k)

    results.append({'file': base, 'f1': f1, 'windowdiff': wd,
                    'n_gt': sum(ref), 'n_pred': sum(hyp)})
    print(f"{base:35s}  F1={f1:.3f}  WD={wd:.3f}  "
          f"(gt_boundaries={sum(ref)}, pred_boundaries={sum(hyp)})")

# -- summary --

if results:
    mean_f1 = np.mean([r['f1']         for r in results])
    mean_wd = np.mean([r['windowdiff']  for r in results])
    print(f"\n{'='*60}")
    print(f"Files evaluated : {len(results)}")
    print(f"Mean F1         : {mean_f1:.3f}")
    print(f"Mean WindowDiff : {mean_wd:.3f}")
else:
    print("No files evaluated.")


[warn] 1719264285.rec.xml: 516 chunks, truncating to 100
1719264285                           F1=0.286  WD=0.352  (gt_boundaries=4, pred_boundaries=3)
[warn] 1719524529.rec.xml: 419 chunks, truncating to 100
1719524529                           F1=0.400  WD=0.173  (gt_boundaries=2, pred_boundaries=3)
[warn] 1721166491.rec.xml: 8103 chunks, truncating to 100
1721166491                           F1=0.000  WD=0.489  (gt_boundaries=4, pred_boundaries=2)
[warn] 1721203635.rec.xml: 6598 chunks, truncating to 100
1721203635                           F1=0.500  WD=0.120  (gt_boundaries=2, pred_boundaries=2)
1721231136                           F1=0.667  WD=0.133  (gt_boundaries=2, pred_boundaries=1)
[warn] 1721655754.rec.xml: 613 chunks, truncating to 100
1721655754                           F1=0.400  WD=0.286  (gt_boundaries=3, pred_boundaries=2)
1721772359                           F1=0.500  WD=0.500  (gt_boundaries=2, pred_boundaries=2)
[warn] 1721946123.rec.xml: 322 chunks, truncating to 10